In [2]:
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np

In [3]:
from pathlib import Path
from dotenv import load_dotenv
import os
load_dotenv("../.env")

True

In [4]:
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
DAGSHUB_REPO_OWNER = os.getenv("DAGSHUB_REPO_OWNER")
DAGSHUB_REPO_NAME = os.getenv("DAGSHUB_REPO_NAME")

In [5]:
df = pd.read_csv('IMDB.csv')
df = df.sample(500)
df.to_csv('data.csv', index=False)
df.head()

,review,sentiment
678,"I was prepared for a bad movie, and a bad movi...",negative
900,"A family (mother-Patricia Clarkson, father-Jak...",positive
844,"Repugnant Bronson thriller. Unfortunately, it'...",negative
282,i love bad shark movies. i really do. i laugh ...,negative
528,I have seen previous movies from Cédric Klapis...,negative


In [6]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['review'] = df['review'].apply(lower_case)
        df['review'] = df['review'].apply(remove_stop_words)
        df['review'] = df['review'].apply(removing_numbers)
        df['review'] = df['review'].apply(removing_punctuations)
        df['review'] = df['review'].apply(removing_urls)
        df['review'] = df['review'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

In [7]:
df = normalize_text(df)
df.head()

,review,sentiment
678,prepared bad movie bad movie is guess complain...,negative
900,family mother patricia clarkson father jake we...,positive
844,repugnant bronson thriller unfortunately techn...,negative
282,love bad shark movie really do laugh hysterica...,negative
528,seen previous movie cédric klapisch therefore ...,negative


In [8]:
df['sentiment'].value_counts()

sentiment
negative    270
positive    230
Name: count, dtype: int64

In [9]:
x = df['sentiment'].isin(['positive','negative'])
df = df[x]

In [10]:
df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})
df.head()

,review,sentiment
678,prepared bad movie bad movie is guess complain...,0
900,family mother patricia clarkson father jake we...,1
844,repugnant bronson thriller unfortunately techn...,0
282,love bad shark movie really do laugh hysterica...,0
528,seen previous movie cédric klapisch therefore ...,0


In [11]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [12]:
vectorizer = CountVectorizer(max_features=50)
X = vectorizer.fit_transform(df['review'])
y = df['sentiment']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
import dagshub
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

dagshub.init(
    repo_owner=DAGSHUB_REPO_OWNER,
    repo_name=DAGSHUB_REPO_NAME,
    mlflow=True
)

mlflow.set_experiment("Logistic Regression Baseline")

Accessing as tarun931

Initialized MLflow to track repo "tarun931/MLOps-Capstone-Project"

Repository tarun931/MLOps-Capstone-Project initialized!

<Experiment: artifact_location='mlflow-artifacts:/ccd022908eae4f2da204152b51709e76', creation_time=1784475715496, experiment_id='0', last_update_time=1784475715496, lifecycle_stage='active', name='Logistic Regression Baseline', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [15]:
import mlflow
import logging
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

logging.info("Starting MLflow run...")

with mlflow.start_run():
    start_time = time.time()
    
    try:
        logging.info("Logging preprocessing parameters...")
        mlflow.log_param("vectorizer", "Bag of Words")
        mlflow.log_param("num_features", 50)
        mlflow.log_param("test_size", 0.2)

        logging.info("Initializing Logistic Regression model...")
        model = LogisticRegression(max_iter=1000)  # Increase max_iter to prevent non-convergence issues

        logging.info("Fitting the model...")
        model.fit(X_train, y_train)
        logging.info("Model training complete.")

        logging.info("Logging model parameters...")
        mlflow.log_param("model", "Logistic Regression")

        logging.info("Making predictions...")
        y_pred = model.predict(X_test)

        logging.info("Calculating evaluation metrics...")
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        logging.info("Logging evaluation metrics...")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)

        logging.info("Saving and logging the model...")
        mlflow.sklearn.log_model(model, "model")

        # Log execution time
        end_time = time.time()
        logging.info(f"Model training and logging completed in {end_time - start_time:.2f} seconds.")

        # Save and log the notebook
        # notebook_path = "exp1_baseline_model.ipynb"
        # logging.info("Executing Jupyter Notebook. This may take a while...")
        # os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
        # mlflow.log_artifact(notebook_path)

        # logging.info("Notebook execution and logging complete.")

        # Print the results for verification
        logging.info(f"Accuracy: {accuracy}")
        logging.info(f"Precision: {precision}")
        logging.info(f"Recall: {recall}")
        logging.info(f"F1 Score: {f1}")

    except Exception as e:
        logging.error(f"An error occurred: {e}", exc_info=True)


2026-07-20 03:16:58,521 - INFO - Starting MLflow run...
2026-07-20 03:16:59,031 - INFO - Logging preprocessing parameters...
2026-07-20 03:16:59,967 - INFO - Initializing Logistic Regression model...
2026-07-20 03:16:59,968 - INFO - Fitting the model...
2026-07-20 03:16:59,982 - INFO - Model training complete.
2026-07-20 03:16:59,982 - INFO - Logging model parameters...
2026-07-20 03:17:00,284 - INFO - Making predictions...
2026-07-20 03:17:00,286 - INFO - Calculating evaluation metrics...
2026-07-20 03:17:00,294 - INFO - Logging evaluation metrics...
2026-07-20 03:17:01,421 - INFO - Saving and logging the model...
2026/07/20 03:17:05 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when logging the model.
2026-07-20 03:17:05,307 - INFO - Model training and logging completed in 6.28 seconds.
2026-07-20 03:17:05,308 - INFO - Accuracy: 0.69
2026-07-20 03:17:05,308 - INFO - Precision: 0.5384615384615384
2026-07-2